In [6]:
import pandas as pd
import numpy as np

# 1. LOAD MASTER DATA
try:
    df = pd.read_csv("Final_Auction_Master.csv")
    print(f"Loaded Master Dataset: {len(df)} players.")
except FileNotFoundError:
    print("Error: Final_Auction_Master.csv not found.")
    exit()

# 2. CLEAN DATA ISSUES
# Fix the "99" anomaly for non-bowlers/non-batters
df['Bowl_Avg'] = df['Bowl_Avg'].replace(99.0, np.nan)
df['Eco_Median'] = df['Eco_Median'].replace(12.0, np.nan) # Assuming 12 was the fillna value
df['SR_Median'] = df['SR_Median'].replace(0, np.nan)

# 3. DEFINE ROLES (Strict)
def get_role(row):
    if row['Is_Keeper'] == 1: return 'Wicketkeeper'
    
    # Bowlers (Played > 10 innings)
    if row['Bowl_Innings'] > 5:
        style = str(row['Bowl_Style']).lower()
        if 'spin' in style or 'leg' in style or 'orthodox' in style or 'break' in style:
            return 'Spinner'
        if 'fast' in style or 'medium' in style:
            return 'Pacer'
            
    # Specialist Batters vs All-Rounders
    if row['Bat_Innings'] > 5:
        if row['Bowl_Innings'] > 5 and row['Wickets'] > 5:
            return 'All-Rounder'
        return 'Batter'
    
    return 'Backup'

df['Role'] = df.apply(get_role, axis=1)

# 4. "PROVEN WINNER" SCORING ALGORITHM
def calculate_proven_score(row):
    score = 0
    
    # A. MARKET VALIDATION (The biggest factor)
    # If sold in 2025, they are "proven".
    if row['Sold_2025']:
        score += 50  # Huge bonus for being a sold player
        # Add points for high price (capped at 20)
        score += min(row['Past_Price_2025'] * 2, 30)
        
    # B. PERFORMANCE STATS
    # Batting
    if row['Bat_Innings'] > 5:
        if row['SR_Median'] > 135: score += 15
        if row['Bat_Avg'] > 30: score += 10
    
    # Bowling
    if row['Bowl_Innings'] > 5:
        if row['Wickets'] > 20: score += 15
        if row['Eco_Median'] < 8.0: score += 15
        
    # C. AGE FACTOR
    if row['Age'] < 35: score += 5
            
    return score

df['Proven_Score'] = df.apply(calculate_proven_score, axis=1)

# 5. SELECTION LOGIC (Purse: 16.05 Cr, 1 Overseas)
available = df[df['Retained'] == False].copy()
selected_squad = []
current_spend = 0.0
overseas_count = 0

def pick_player(role_filter, top_n=1, budget_cap=16.0, must_be_sold=False, allow_overseas=False):
    global current_spend, overseas_count
    
    # Filter
    candidates = available[
        (available['Role'].str.contains(role_filter, case=False, na=False)) &
        (~available['Player Name'].isin([x['Name'] for x in selected_squad]))
    ].sort_values('Proven_Score', ascending=False)
    
    if must_be_sold:
        candidates = candidates[candidates['Sold_2025'] == True]

    added = 0
    for _, row in candidates.iterrows():
        if added >= top_n: break
        
        is_os = row['Category'] == 'Overseas'
        if is_os and (not allow_overseas or overseas_count >= 1): continue
        
        # Smart Bid: Base Price + (Past Price * 0.8). 
        # We don't want to overpay, but we need to be realistic.
        # Floor is Base Price. Cap is Budget Cap.
        market_est = row['Past_Price_2025'] if row['Past_Price_2025'] > 0 else row['Base Price']
        bid = max(row['Base Price'], min(market_est, budget_cap))
        
        if current_spend + bid > 16.05: continue
        
        selected_squad.append({
            'Name': row['Player Name'],
            'Role': row['Role'],
            'Type': row['Category'],
            'Past_Price': row['Past_Price_2025'],
            'Bid': bid,
            'Score': row['Proven_Score']
        })
        current_spend += bid
        if is_os: overseas_count += 1
        added += 1

# --- EXECUTE STRATEGY ---

# 1. THE BIG SPINNER (Hasaranga/Theekshana)
# High budget. Overseas allowed. Must be a "Sold" player.
pick_player('Spinner', top_n=1, budget_cap=8.0, must_be_sold=True, allow_overseas=True)

# 2. THE ANCHOR / TOP ORDER (Tripathi/Venky Iyer)
# Indian. High budget.
pick_player('Batter|All-Rounder', top_n=1, budget_cap=5.0, must_be_sold=True, allow_overseas=False)

# 3. THE FINISHER / ALL-ROUNDER
pick_player('All-Rounder|Batter', top_n=1, budget_cap=3.0, must_be_sold=True, allow_overseas=False)

# 4. BACKUP KEEPER
pick_player('Wicketkeeper', top_n=1, budget_cap=1.0, must_be_sold=True, allow_overseas=False)

# 5. FAST BOWLER (Depth)
pick_player('Pacer', top_n=1, budget_cap=1.0, must_be_sold=False, allow_overseas=False)

# 6. FILL REMAINING (Best Available Value)
# We need at least 2 players. If we have 5, we can add more cheap ones.
remaining_slots = 9 - len(selected_squad)
for i in range(remaining_slots):
    if current_spend > 15.5: break
    pick_player('Backup|Batter|Pacer', top_n=1, budget_cap=0.5, must_be_sold=False, allow_overseas=False)

# OUTPUT
final_df = pd.DataFrame(selected_squad)
print("\n" + "="*50)
print(f"FINAL PROVEN SQUAD (Total Cost: {current_spend:.2f} Cr)")
print("="*50)
print(final_df[['Name', 'Role', 'Type', 'Past_Price', 'Bid', 'Score']])

Loaded Master Dataset: 350 players.

FINAL PROVEN SQUAD (Total Cost: 8.20 Cr)
             Name          Role      Type  Past_Price  Bid  Score
0   Glenn Maxwell       Spinner  Overseas        4.20  4.2  103.4
1    Raghav Goyal   All-Rounder    Indian        0.00  0.3   85.0
2   Atharva Taide        Batter    Indian        0.30  0.3   80.6
3    Harvik Desai  Wicketkeeper    Indian        0.00  0.3   80.0
4  Venkatesh Iyer         Pacer    Indian       23.75  1.0  100.0
5      Akash Deep         Pacer    Indian        8.00  1.0   86.0
6     Shivam Mavi         Pacer    Indian        0.00  0.5   85.0
7         KM Asif         Pacer    Indian        0.00  0.3   85.0
8    Basil Thampi         Pacer    Indian        0.00  0.3   85.0
